# Geocoder example — Norwegian towns (Nominatim)

Small, self-contained demo of the pattern in `geocoding/geocoder.ipynb`,
fixed to the free Nominatim backend and the five-town example dataset.

Run `make clean-notebook` first to produce `data/output/example_clean.csv`,
or just run `make geocoder-notebook` — it does that for you.

# Geocoder

One notebook, two interchangeable backends.

| | Nominatim (OpenStreetMap) | Google Geocoding API |
|---|---|---|
| Cost | Free | 10,000 free calls/month, then $5.00 / 1,000 |
| Rate | 1 request/second (enforced) | ~50 req/s (throttled to 5 here) |
| Setup | `NOMINATIM_USER_AGENT` in `.env` | `GOOGLE_MAPS_API_KEY` **and** `GOOGLE_GEOCODING_CONFIRM=1` |

Every result is cached in `.cache/geocode.sqlite`, so re-running this notebook
costs **zero** provider calls. `MAX_REQUESTS_PER_RUN` caps live calls per run.

In [ ]:
import pandas as pd

from geocoding_tool import (
    build_query,
    geocode_dataframe,
    get_geocoder,
    load_env,
    project_root,
    to_geodataframe,
    write_attribution,
)

load_env()  # reads .env (copy .env.example first; set NOMINATIM_USER_AGENT)

ROOT = project_root()
INPUT_PATH = ROOT / "data" / "output" / "example_clean.csv"
OUTPUT_PATH = ROOT / "data" / "output" / "example_geocoded.csv"

geocoder = get_geocoder("nominatim")
print(f"provider: {geocoder.name}")
print(f"budget:   {geocoder.budget.limit} live calls this run")
print(f"cache:    {len(geocoder.cache)} results already stored")

## Read data

In [ ]:
df = pd.read_csv(INPUT_PATH, dtype="string")
df["query"] = build_query(df, ["town"], suffix="Norway")
df

## Geocode

Nominatim is rate limited to 1 request/second by policy, so five unique
towns take a few seconds.

In [ ]:
result = geocode_dataframe(df, "query", geocoder)
result

## Inspect & export

In [ ]:
gdf = to_geodataframe(result)

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
result.to_csv(OUTPUT_PATH, index=False)
gdf.to_file(OUTPUT_PATH.with_suffix(".gpkg"), driver="GPKG")
sidecar = write_attribution(OUTPUT_PATH, geocoder)

print(f"wrote {OUTPUT_PATH.relative_to(ROOT)}")
print(f"wrote {OUTPUT_PATH.with_suffix('.gpkg').relative_to(ROOT)}")
print(f"wrote {sidecar.relative_to(ROOT)}")
gdf